# 14 — Final OOF Ensemble Robustness & Pipeline Lock

Mục tiêu: **không train thêm**. Dùng OOF predictions hiện có để chọn một ensemble weight + threshold ổn định, tránh overfit vào đúng một điểm cực đại.

Nguồn:
- grayscale DenseNet seed 2026
- grayscale DenseNet seed 3407

Best OOF trước bước này:
- weight 2026 = 0.73
- weight 3407 = 0.27
- threshold = 0.620
- P = 0.9870215029

TTA đã bị loại vì làm ensemble giảm điểm.

Notebook này:
1. quét weight/threshold quanh vùng tốt
2. tính P toàn OOF và P từng fold
3. đánh giá plateau/stability
4. chọn cấu hình **robust lock**
5. lưu `locked_pipeline.json`

Holdout vẫn chưa dùng.


In [ ]:
print('>>> CELL 1 START: load OOF', flush=True)
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix,f1_score,roc_auc_score

ROOT=Path(r'D:\medicine')
P26=ROOT/'cv_outputs'/'densenet121_224_gray_b8a2'/'oof_predictions.csv'
P34=ROOT/'cv_outputs'/'densenet121_224_gray_b8a2_seed3407'/'oof_predictions.csv'
OUT=ROOT/'cv_outputs'/'final_ensemble_lock'
OUT.mkdir(parents=True,exist_ok=True)

a=pd.read_csv(P26)[['filename','y_true','p_pneumonia','fold']].rename(columns={'p_pneumonia':'p26'})
b=pd.read_csv(P34)[['filename','y_true','p_pneumonia','fold']].rename(columns={'p_pneumonia':'p34','fold':'fold34'})
df=a.merge(b,on=['filename','y_true'],validate='one_to_one')
assert len(df)==4362
assert (df.fold==df.fold34).all()
df=df.drop(columns='fold34')

print('OOF rows:',len(df),flush=True)
print('Folds:',df.fold.value_counts().sort_index().to_dict(),flush=True)


In [ ]:
print('>>> CELL 2 START: metric helpers', flush=True)

def score(y,p,t,auc=None):
    y=np.asarray(y,int);p=np.asarray(p,float)
    pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y)
    se=tp/max(tp+fn,1)
    sp=tn/max(tn+fp,1)
    f1=f1_score(y,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    if auc is None: auc=roc_auc_score(y,p)
    return {'P':float(P),'Accuracy':float(acc),'Sensitivity':float(se),
            'Specificity':float(sp),'F1':float(f1),'AUROC':float(auc),
            'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
            'threshold':float(t)}

y=df.y_true.values


In [ ]:
print('>>> CELL 3 START: dense robustness grid', flush=True)

rows=[]
weights=np.round(np.arange(.55,.901,.005),3)
thresholds=np.round(np.arange(.50,.701,.001),3)

for w26 in weights:
    w34=1.0-w26
    p=w26*df.p26.values+w34*df.p34.values
    auc=float(roc_auc_score(y,p))

    for t in thresholds:
        m=score(y,p,float(t),auc)
        fold_ps=[]
        for f in range(5):
            mask=df.fold.values==f
            fold_ps.append(score(y[mask],p[mask],float(t))['P'])

        rows.append({
            'w26':float(w26),'w34':float(w34),'threshold':float(t),
            **m,
            'fold_mean':float(np.mean(fold_ps)),
            'fold_std':float(np.std(fold_ps,ddof=1)),
            'fold_min':float(np.min(fold_ps)),
            'fold_max':float(np.max(fold_ps)),
        })

grid=pd.DataFrame(rows)
grid.to_csv(OUT/'robustness_grid.csv',index=False)

best=grid.sort_values(['P','fold_std'],ascending=[False,True]).iloc[0]
print('Global max:',best.to_dict(),flush=True)


In [ ]:
print('>>> CELL 4 START: plateau / robust selection', flush=True)

best_P=float(grid.P.max())

# Near-optimal plateau: within 0.00030 of global max.
plateau=grid[grid.P >= best_P-0.00030].copy()

# Prefer configs that are near-optimal but stable across folds.
# Robust rank: highest fold_min, then lowest fold_std, then closest to simple weights.
plateau['distance_from_070_030']=abs(plateau.w26-.70)

robust=plateau.sort_values(
    ['fold_min','fold_std','distance_from_070_030','P'],
    ascending=[False,True,True,False]
).iloc[0]

plateau.sort_values(['fold_min','fold_std'],ascending=[False,True]).head(100).to_csv(
    OUT/'near_optimal_plateau.csv',index=False
)

print('Best P:',best_P,flush=True)
print('Plateau configs:',len(plateau),flush=True)
print('Robust choice:',robust.to_dict(),flush=True)

# Evaluate exact previously-found 0.73 / 0.27 / 0.62.
p_prev=.73*df.p26.values+.27*df.p34.values
prev=score(y,p_prev,.620,float(roc_auc_score(y,p_prev)))
prev_fold=[]
for f in range(5):
    mask=df.fold.values==f
    prev_fold.append(score(y[mask],p_prev[mask],.620)['P'])

print('Previous best config:',prev,flush=True)
print('Previous fold P:',prev_fold,flush=True)


In [ ]:
print('>>> CELL 5 START: lock pipeline', flush=True)

w26=float(robust.w26)
w34=float(robust.w34)
t=float(robust.threshold)
p=w26*df.p26.values+w34*df.p34.values
auc=float(roc_auc_score(y,p))
m=score(y,p,t,auc)

fold_metrics=[]
for f in range(5):
    mask=df.fold.values==f
    fm=score(y[mask],p[mask],t)
    fold_metrics.append({'fold':int(f),**fm})

locked={
    'status':'LOCKED_BEFORE_HOLDOUT',
    'architecture':'DenseNet121',
    'image_size':224,
    'preprocessing':'force_grayscale_then_rgb',
    'models':[
        {
            'name':'gray_seed2026',
            'weight':w26,
            'checkpoint_dir':str(ROOT/'cv_outputs'/'densenet121_224_gray_b8a2')
        },
        {
            'name':'gray_seed3407',
            'weight':w34,
            'checkpoint_dir':str(ROOT/'cv_outputs'/'densenet121_224_gray_b8a2_seed3407')
        }
    ],
    'tta':False,
    'threshold':t,
    'selection_rule':'near-optimal OOF plateau within 0.00030 of max P; maximize worst-fold P, then minimize fold std',
    'oof_metrics':m,
    'fold_metrics':fold_metrics,
    'global_grid_max_P':best_P,
    'global_grid_max_config':{
        'w26':float(best.w26),
        'w34':float(best.w34),
        'threshold':float(best.threshold),
        'P':float(best.P),
        'fold_std':float(best.fold_std),
        'fold_min':float(best.fold_min)
    },
    'holdout_used':False
}

with open(OUT/'locked_pipeline.json','w',encoding='utf-8') as f:
    json.dump(locked,f,indent=2)

pd.DataFrame(fold_metrics).to_csv(OUT/'locked_pipeline_fold_metrics.csv',index=False)

print(json.dumps(locked,indent=2),flush=True)


## Sau notebook này

Nếu robust lock hợp lý, pipeline sẽ được **đóng băng**. Bước sau đó mới là đánh giá internal sealed holdout đúng một lần.

Không được thay weight/threshold dựa trên holdout sau khi mở.
